# Minería de Texto · Unidad I

## Práctica guiada en Python

En esta práctica voy a limpiar tres comentarios, separar las palabras, contar el vocabulario, quitar stopwords y revisar las frecuencias. También probaré la limpieza con un texto que contiene HTML.

## 1. Preparar el entorno y el corpus

In [ ]:
import re
from collections import Counter

corpus = [
    "¡El servicio al cliente es excelente y muy rápido en la entrega!",
    "El servicio es pésimo y la atención al cliente es muy lenta.",
    "Excelente atención pero el servicio de entrega es lento.",
]

print("Librerías listas")
for i, d in enumerate(corpus, start=1):
    print(f"D{i}: {d}")

## 2. Limpiar HTML y ruido

Primero quito etiquetas HTML, enlaces, correos y emojis. Después normalizo los espacios.

In [ ]:
def limpiar_html_y_ruido(texto):
    texto = re.sub(r"<[^>]+>", "", texto)  # etiquetas HTML
    texto = re.sub(r"https?://\S+", "", texto)  # URLs
    texto = re.sub(r"\S+@\S+", "", texto)  # correos
    texto = re.sub(r"[\U0001F300-\U0001FAFF\u2600-\u27BF]", "", texto)  # emojis
    texto = re.sub(r"\s+", " ", texto).strip()  # espacios repetidos
    return texto

sucio = "<p>¡El <b>servicio</b> al cliente es excelente!</p> Visítanos https://ejemplo.com info@ejemplo.com ☺"
print("Antes:", sucio)
print("Después:", limpiar_html_y_ruido(sucio))

corpus = [limpiar_html_y_ruido(d) for d in corpus]

## 3. Normalizar: minúsculas y puntuación

In [ ]:
def normalizar(texto):
    texto = limpiar_html_y_ruido(texto)
    texto = texto.lower()
    texto = re.sub(r"[¡!¿?.,;:]", "", texto)
    return texto

docs_norm = [normalizar(d) for d in corpus]
for i, d in enumerate(docs_norm, start=1):
    print(f"d{i} = {d}")

assert "pésimo" in docs_norm[1]  # los acentos se conservan

## 4. Tokenizar y contar los tokens

Con `split()` separo cada comentario en palabras.

In [ ]:
tokens = [d.split() for d in docs_norm]
for i, t in enumerate(tokens, start=1):
    print(f"T{i} = {t}")

n = [len(t) for t in tokens]
N = sum(n)
print("n =", n, " N =", N)

## 5. Construir el vocabulario

El vocabulario contiene las palabras diferentes del corpus. Uso conjuntos para quitar repetidas.

In [ ]:
print(len(tokens[1]), len(set(tokens[1])))  # lista vs. conjunto

V = set(tokens[0])
for t in tokens[1:]:
    V = V | set(t)

V = sorted(V)
print(V)
print("|V| =", len(V))
print("Repetidos descartados:", N - len(V))

## 6. Quitar stopwords

Uso el conjunto de palabras indicado en la práctica.

In [ ]:
S = {"al", "de", "el", "en", "es", "la", "muy", "pero", "y"}
print("¿S está dentro de V?", S <= set(V))

V_filtrado = sorted(set(V) - S)
print("Vocabulario filtrado:", V_filtrado)
print("|S| =", len(S), " |V_filtrado| =", len(V_filtrado))
print("% stopwords =", len(S) / len(V) * 100)

## 7. Contar frecuencias (TF)

Cuento los términos filtrados en cada comentario y después sumo sus frecuencias.

In [ ]:
vf = set(V_filtrado)
tf = [Counter(w for w in t if w in vf) for t in tokens]
for i, c in enumerate(tf, start=1):
    print(f"TF(D{i}) =", dict(c))

tf_total = sum(tf, Counter())
print("Frecuencia total:", tf_total.most_common())

## 8. Ficha de métricas y comprobaciones

In [ ]:
ficha = {
    "N (tokens totales)": N,
    "|V| (vocabulario)": len(V),
    "|S| (stopwords)": len(S),
    "|V_filtrado|": len(V_filtrado),
}
for nombre, valor in ficha.items():
    print(f"{nombre:<20} {valor}")

assert N == 33
assert len(V) == 18
assert len(S) == 9
assert len(S) / len(V) == 0.5
assert len(V_filtrado) == len(V) - len(S)
print("Todas las verificaciones pasaron")

## 9. Gráfica de frecuencias

In [ ]:
import matplotlib.pyplot as plt

frecuencias = tf_total.most_common()
palabras = [item[0] for item in frecuencias]
cantidades = [item[1] for item in frecuencias]

plt.figure(figsize=(9, 4))
plt.bar(palabras, cantidades)
plt.title("Frecuencia total por término")
plt.xlabel("Término")
plt.ylabel("Frecuencia")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 10. Retos opcionales

### Reto 1. Agregar un cuarto comentario con HTML

Lo pruebo aparte para no cambiar las métricas del corpus principal.

In [ ]:
comentario_extra = "<p>La <b>atención</b> fue buena y la entrega rápida.</p>"
extra_limpio = normalizar(comentario_extra)
print("Comentario extra normalizado:", extra_limpio)
print("Tokens del comentario extra:", extra_limpio.split())
print("Si se agrega al corpus, las métricas N, V y V_filtrado se vuelven a calcular.")

### Reto 2. Comparar con stopwords de NLTK

Esta parte descarga el recurso de stopwords en español. En Colab se puede ejecutar directamente.

In [ ]:
import nltk
nltk.download("stopwords")
from nltk.corpus import stopwords

stopwords_es = set(stopwords.words("spanish"))
tokens_nltk = [w for t in tokens for w in t if w not in stopwords_es]
print("Cantidad de stopwords de NLTK:", len(stopwords_es))
print("Tokens restantes con NLTK:", tokens_nltk)

### Reto 3. Aplicar stemming en español

In [ ]:
from nltk.stem.snowball import SnowballStemmer

stemmer = SnowballStemmer("spanish")
V_stem = sorted({stemmer.stem(palabra) for palabra in V_filtrado})
print("Vocabulario original:", V_filtrado)
print("Vocabulario con stemming:", V_stem)
print("Tamaño después del stemming:", len(V_stem))
assert len(V_stem) == 8

## Conclusión

Con esta práctica convertí comentarios de texto en datos que puedo contar y comparar. La limpieza ayuda a quitar ruido, las stopwords reducen palabras poco útiles y las frecuencias muestran los términos más repetidos. El resultado principal coincide con las métricas indicadas: 33 tokens, 18 palabras únicas, 9 stopwords y 9 términos filtrados.